# 2023–2026韩国海关前20日半导体出口同比：留出数值 v2

v1预览断言在7份HWP后失败；v2只修复预览支持计数为30/30，固定原始字节与数值语法不变。


In [ ]:
import hashlib,json,os,pathlib,re,zlib,zipfile,xml.etree.ElementTree as ET,shlex
import numpy as np,pandas as pd,qlib,olefile,pypdf
from pypdf import PdfReader
from qlib.workflow import R
from src.data.nas_market_data import archive_nas_recorder,_settings,_ssh
root=pathlib.Path.cwd();card=root/'research/kcs_early20_2023_2026_semiconductor_yoy_source_v2_20261023';plan=json.loads((card/'plan.json').read_text());src=plan['sources'];sha=lambda p:hashlib.sha256(pathlib.Path(p).read_bytes()).hexdigest();assert plan['frozen'] and not (card/'run_state.json').exists() and olefile.__version__=='0.47' and pypdf.__version__=='6.20.1'
old=root/src['held_capture_card'];repair=root/src['August_repair_card'];train=root/src['training_numeric_card'];assert sha(old/'summary.json')==src['held_capture_summary_sha256'] and sha(repair/'summary.json')==src['August_repair_summary_sha256'] and sha(train/'summary.json')==src['training_numeric_summary_sha256']
snapshot=root/src['snapshot_root'];rs=[]
for year,h in src['year_manifest_sha256'].items():
 p=snapshot/year/'year_manifest.json';assert sha(p)==h;rs.extend(json.loads(p.read_text())['months'])
assert len(rs)==45;rp=root/src['August_repair_manifest'];assert sha(rp)==src['August_repair_manifest_sha256'];aug=json.loads(rp.read_text());assert aug['new_month_result']['all_linked_signatures_qualified'];rs=[m if m['month']!='2025-08' else {'month':'2025-08','page':aug['new_page']} for m in rs];assert all(m['page'] and m['page']['attachments'] for m in rs)
pat=re.compile(r'반도체\s*\(\s*([△▽▲▼−\-+]?\s*[\d,.]+)\s*%\s*[↑↓]?\s*\)');anchor=re.compile('^'+pat.pattern);label='반도체'.encode('utf-16le');export='수출현황'.encode('utf-16le');imported='수입현황'.encode('utf-16le')
def value(token):
 s=re.sub(r'\s+','',token);negative=s[:1] in ('△','▽','▼','−','-');s=s.lstrip('△▽▲▼−-+').replace(',','');v=float(s);assert np.isfinite(v);return -v if negative else v
def span(text):
 start=text.find('수출현황');end=text.find('수입현황',start+4) if start>=0 else -1;assert start>=0;return text[start:end if end>=0 else None],start,end
state={'status':'source_numeric_reads_started','plan_sha256':sha(card/'plan.json'),'HWP_reads':0,'HWPX_reads':0,'PDF_reads':0,'HWP_preview_agree':0,'HWPX_PDF_agree':0,'model_fits':0,'native_accounts':0};(card/'run_state.json').write_text(json.dumps(state,indent=2)+'\n');rows=[];details=[]
for month in rs:
 key=month['month'];files=month['page']['attachments'];hwps=[x for x in files if x['ext']=='hwp'];hxs=[x for x in files if x['ext']=='hwpx'];pdfs=[x for x in files if x['ext']=='pdf'];assert len(hwps)+len(hxs)==1 and all(a['signature_ok'] for a in files);preview_token=None;PDF_token=None;PDF_sha=None;location={};meta={};format='hwp' if hwps else 'hwpx'
 if hwps:
  a=hwps[0];p=root/a['local_path'];assert sha(p)==a['body_sha256'] and p.stat().st_size==a['body_bytes'] and olefile.isOleFile(str(p));state['HWP_reads']+=1;(card/'run_state.json').write_text(json.dumps(state,indent=2)+'\n')
  with olefile.OleFileIO(str(p)) as doc:
   assert doc.exists('BodyText/Section0') and doc.exists('PrvText');body=zlib.decompress(doc.openstream('BodyText/Section0').read(),-15);preview=doc.openstream('PrvText').read().decode('utf-16le','replace');md=doc.get_metadata();meta={'HWP_created':str(md.create_time) if md.create_time else None,'HWP_last_saved':str(md.last_saved_time) if md.last_saved_time else None}
  start=body.find(export);end=body.find(imported,start+len(export)) if start>=0 else -1;assert start>=0 and (end<0 or end>start);tokens=[];at=0
  while True:
   at=body.find(label,at)
   if at<0:break
   if at>start and (end<0 or at<end):
    hit=anchor.search(body[at:at+140].decode('utf-16le','replace'))
    if hit:tokens.append({'offset':at,'raw':hit.group(1)})
   at+=len(label)
  assert len(tokens)==1,(key,tokens);token=tokens[0]['raw'];location={'export_heading_byte':start,'import_heading_byte':end,'label_byte':tokens[0]['offset'],'body_decompressed_bytes':len(body)};text,ps,pe=span(preview);found=pat.findall(text)
  assert len(found)==1 and abs(value(found[0])-value(token))<1e-9,(key,found,token);preview_token=found[0];state['HWP_preview_agree']+=1
  location.update(preview_export_char=ps,preview_import_char=pe,preview_chars=len(preview))
 else:
  a=hxs[0];p=root/a['local_path'];assert sha(p)==a['body_sha256'] and p.stat().st_size==a['body_bytes'] and zipfile.is_zipfile(p);state['HWPX_reads']+=1;(card/'run_state.json').write_text(json.dumps(state,indent=2)+'\n')
  with zipfile.ZipFile(p) as z:assert 'Contents/section0.xml' in z.namelist() and 'Contents/content.hpf' in z.namelist();xml=z.read('Contents/section0.xml');hpf=z.read('Contents/content.hpf')
  rootxml=ET.fromstring(xml);text=''.join(n.text or '' for n in rootxml.iter() if n.tag.endswith('}t'));segment,ps,pe=span(text);matches=pat.findall(segment);assert len(matches)==1,(key,matches);token=matches[0];location={'export_heading_char':ps,'import_heading_char':pe,'section0_xml_sha256':hashlib.sha256(xml).hexdigest(),'HWPX_text_chars':len(text)};meta={'HWPX_content_hpf_sha256':hashlib.sha256(hpf).hexdigest()}
  assert len(pdfs)==1;pdf=pdfs[0];pdfpath=root/pdf['local_path'];assert sha(pdfpath)==pdf['body_sha256'] and pdfpath.stat().st_size==pdf['body_bytes'];state['PDF_reads']+=1;(card/'run_state.json').write_text(json.dumps(state,indent=2)+'\n');alltext='\n'.join((page.extract_text() or '') for page in PdfReader(str(pdfpath)).pages);segment,pp,pi=span(alltext);found=pat.findall(segment);assert len(found)==1 and abs(value(found[0])-value(token))<1e-9,(key,found,token);PDF_token=found[0];PDF_sha=pdf['body_sha256'];state['HWPX_PDF_agree']+=1;location.update(PDF_export_char=pp,PDF_import_char=pi,PDF_text_chars=len(alltext))
  if key=='2025-08':assert a['local_path'].startswith('data/snapshots/kcs_early20_2025_aug_transport_repair_20261023/')
  else:assert a['local_path'].startswith(src['snapshot_root'])
 rows.append({'period':key,'reported_semiconductor_export_yoy_pct':value(token),'raw_token':token,'source_format':format,'source_sha256':a['body_sha256'],'source_bytes':a['body_bytes'],'HWP_preview_corroborated':preview_token is not None,'PDF_corroborated':PDF_token is not None,'PDF_sha256':PDF_sha,'source_vintage_first_public_qualified':False})
 details.append({'period':key,'source_format':format,'source_file':a['local_path'],'source_sha256':a['body_sha256'],'raw_token':token,'preview_token':preview_token,'PDF_token':PDF_token,'location':location,'metadata':meta})
assert len(rows)==len(details)==45 and state['HWP_reads']==30 and state['HWPX_reads']==state['PDF_reads']==state['HWPX_PDF_agree']==15 and state['HWP_preview_agree']==30;table=pd.DataFrame(rows);assert table.period.tolist()==[f'{y}-{m:02d}' for y in (2023,2024,2025) for m in range(1,13)]+[f'2026-{m:02d}' for m in range(1,10)]
result={'status':'official_2023_2026_KCS_early20_reported_semiconductor_YoY_numeric_held_complete_PIT_pending','plan_sha256':sha(card/'plan.json'),'months':45,'HWP_reads':30,'HWPX_reads':15,'PDF_reads':15,'HWP_preview_corroborated':30,'HWPX_PDF_corroborated':15,'HWP_preview_truncated':[],'negative_YoY_months':int(table.reported_semiconductor_export_yoy_pct.lt(0).sum()),'YoY_min':float(table.reported_semiconductor_export_yoy_pct.min()),'YoY_max':float(table.reported_semiconductor_export_yoy_pct.max()),'historical_first_public_vintage_qualified':False,'working_day_adjusted':False,'SOXX_outcome_reads':0,'model_fits':0,'native_accounts':0};(card/'result.json').write_text(json.dumps(result,ensure_ascii=False,indent=2)+'\n')
os.environ['MLFLOW_ALLOW_FILE_STORE']='true';qlib.init(provider_uri=str(root/'data/qlib/minute_information_pit_allocator_20261012/daily_prices'),region='us',kernels=1)
with R.start(experiment_name=plan['experiment'],uri=str(card/'qlib_recorder')):
 rec=R.get_recorder();rec.save_objects(artifact_path='inputs',plan=plan,source_archive_ids=[x['sha256'] for x in json.loads((old/'summary.json').read_text())['year_archives']]+[json.loads((repair/'summary.json').read_text())['source_archive']['sha256']]);rec.save_objects(artifact_path='analysis',monthly_reported_YoY=table,source_offsets_and_crosschecks=details,result=result);rec.log_metrics(source_months=45,HWP_preview_agree=30,HWPX_PDF_agree=15,model_fits=0,native_accounts=0);rid,eid=rec.id,str(rec.experiment_id)
state['status']='finished';state['recorder_id']=rid;state['experiment_id']=eid;(card/'run_state.json').write_text(json.dumps(state,indent=2)+'\n');evidence=archive_nas_recorder(card/'nas_binding.json',workspace_root=root,card=card,recorder_id=rid,experiment_id=eid);assert _ssh(_settings(root)['ssh'],'sha256sum '+shlex.quote(evidence['nas_path'])).split()[0]==evidence['sha256']
summary={'experiment':plan['experiment'],'decision':'KCS_2023_2026_early20_reported_semiconductor_YoY_held_numeric_complete_first_public_PIT_pending','objective_achieved':False,'plan_sha256':result['plan_sha256'],'result':'result.json','recorder_id':rid,'experiment_id':eid,'recorder_uri':str(card/'qlib_recorder'),'NAS_evidence':evidence,'NAS_readback':{'sha256':evidence['sha256'],'verified':True},'monthly_rows':45,'HWP_preview_corroborated':30,'HWPX_PDF_corroborated':15,'SOXX_outcome_reads':0,'model_fits':0,'native_accounts':0,'artifact_policy':{'policy':'slim_card_recorder_first','workflow_config':'workflow.yaml','recorder_only_artifacts':['inputs/*','analysis/monthly_reported_YoY','analysis/source_offsets_and_crosschecks','analysis/result']},'limitations':['Reported raw YoY is working-day sensitive and not demand surprise.','Current HWP/HWPX/PDF bytes and portal dates do not certify historical first-public versions.','No Qlib US-session feature, SOXX outcome or funded Stage1 account yet.']};(card/'summary.json').write_text(json.dumps(summary,ensure_ascii=False,indent=2)+'\n');print(json.dumps({'decision':summary['decision'],'months':45,'HWP_preview_agree':30,'HWPX_PDF_agree':15,'negative':result['negative_YoY_months'],'min':result['YoY_min'],'max':result['YoY_max'],'recorder_id':rid,'NAS_sha256':evidence['sha256']},ensure_ascii=False,indent=2))


## 中文结论

本卡 v1 曾因错误的“2023年7–9月预览截断”断言，在第7份HWP后停止，已单独封存。v2只改该断言，实际读取 **30份HWP、15份HWPX、15份PDF** 和45个月当前原报，同比范围 −44.7% 至 +259.4%；30个HWP正文/预览及15个HWPX/PDF逐值一致。原生Qlib Recorder `7b60cdf8611d4df69354779db6765573` 的NAS `fafe4058d3363c0f346c3be31075e2464eaef2fbbe00d041f0fbeb0c43751cbb` 已冷读三件工件。

共享仓库另有先完成的 [同源卡](../kcs_early20_2023_2026_held_semiconductor_yoy_20261023/conclusion.md)，使用相同的45个月官方当前原件。逐月核对后，**period、数值、原始符号、文件格式、源SHA及字节数六列全部相等，数值最大差0**；精确身份见 [重复读回收据](source_equivalence.json)。因此本卡作为当前已交付的单一数值源入口；另一套仅作为同源读回核对，不当作第二次独立市场验证或另开一条因子研究。v1七次与v2全量来源读取均按实际保留，0 SOXX结果、0拟合、0账户。

这个原报同比仍受韩国工作日/基数影响，历史首次公开字节和美股09:46前可用性未认证。后续只以本卡同一份数值做Qlib时点与原Stage1 39项信息门；即使信息门通过，还需自有状态0bp Tiered账户、2/5bp及独立前向。稳定超过SOXX尚未证明。
